In [3]:
from catboost import CatBoostClassifier, Pool
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import re
import math
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GroupKFold
from sentence_transformers import SentenceTransformer
from scipy.stats import mode

pd.set_option('display.max_columns', None)

In [ ]:
''' 
GroupCV LogReg accuracy: 0.5936
'''

In [10]:
df = pd.read_csv('data.csv').drop('Unnamed: 0', axis=1)
df.head(1)

,transaction_id,terminal_id,customer_id,amount,currency,transaction_time,transaction_hour,transaction_weekday,payment_method,item_count,operator_id,device_id,batch_id,auth_code,response_code,network_type,session_id,reference_number,trace_id,pos_entry_mode,true_mcc,merchant_id,terminal_name,terminal_description,terminal_city,terminal_type,terminal_serial,terminal_version,merchant_name,merchant_city,items_total_price,items_mean_price,items_price_std,items_min_price,items_max_price,items_price_range,items_text,is_weekend,text,amount_log,items_price_cv,items_vs_amount,terminal_name_len,terminal_desc_len,items_text_len,amount_per_item,items_price_skew,is_single_item_expensive,items_vs_amount_gap,price_roundness,item_count_bucket,avg_price_bucket,item_count_vs_price
0,TX00001116,T000233,C000383,272.8,GBP,2024-11-16 22:00:00,22,Saturday,MOBILE,3,OP50,DEV-D873,B7549,771150,0,4G,274b8351,REF16528164,ed39a783-a597-4d5c-884b-8dfa9f4f2083,7,5995,M00119,LOCATION,general cmoupanieon s faovorites service general,HOU,POS,SN819522799,5.6.8,PLAZA002,HOU,144.1,48.033333,16.504619,28.99,58.2,29.21,braenk etuers,1,braenk etuers general cmoupanieon s faovorites...,5.612398,2.910296,0.528226,8,48,13,90.903032,0.211654,0,0.471772,1,1,2,144.1


In [11]:
df['full_text'] = df['terminal_name'].fillna('') + ' ' + \
        df['terminal_description'].fillna('') + ' ' + \
        df['terminal_city'].fillna('') + ' ' + \
        df['items_text'].fillna('')

In [12]:
def normalize_word_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df['full_text'] = df['full_text'].apply(normalize_word_text)

In [505]:
# train_cities = ['PHX','HOU','CHI']  # можно брать уникальные города на train
# df['city_unknown'] = (~df['terminal_city'].isin(train_cities)).astype(int)

# # Фича: generic терминал (содержит цифры или шаблон STORE/SHOP)
# df['terminal_generic'] = df['terminal_name'].str.contains(r'\d+').astype(int)

In [13]:
df['word_text'] = (
    df['terminal_description'].fillna('') + ' ' +
    df['items_text'].fillna('')
)
df['word_text'] = df['word_text'].apply(normalize_word_text)


In [14]:
df["char_text"] = (
    df["terminal_name"].fillna("") + " " +
    df["terminal_description"].fillna("") + " " +
    df["items_text"].fillna("")
)
df["char_text"] = df["char_text"].apply(normalize_word_text)


In [15]:
df

,transaction_id,terminal_id,customer_id,amount,currency,transaction_time,transaction_hour,transaction_weekday,payment_method,item_count,operator_id,device_id,batch_id,auth_code,response_code,network_type,session_id,reference_number,trace_id,pos_entry_mode,true_mcc,merchant_id,terminal_name,terminal_description,terminal_city,terminal_type,terminal_serial,terminal_version,merchant_name,merchant_city,items_total_price,items_mean_price,items_price_std,items_min_price,items_max_price,items_price_range,items_text,is_weekend,text,amount_log,items_price_cv,items_vs_amount,terminal_name_len,terminal_desc_len,items_text_len,amount_per_item,items_price_skew,is_single_item_expensive,items_vs_amount_gap,price_roundness,item_count_bucket,avg_price_bucket,item_count_vs_price,full_text,word_text,char_text
0,TX00001116,T000233,C000383,272.80,GBP,2024-11-16 22:00:00,22,Saturday,MOBILE,3,OP50,DEV-D873,B7549,771150,0,4G,274b8351,REF16528164,ed39a783-a597-4d5c-884b-8dfa9f4f2083,7,5995,M00119,LOCATION,general cmoupanieon s faovorites service general,HOU,POS,SN819522799,5.6.8,PLAZA002,HOU,144.10,48.033333,16.504619,28.99,58.20,29.21,braenk etuers,1,braenk etuers general cmoupanieon s faovorites...,5.612398,2.910296,0.528226,8,48,13,90.903032,0.211654,0,0.471772,1,1,2,144.10,location general cmoupanieon s faovorites serv...,general cmoupanieon s faovorites service gener...,location general cmoupanieon s faovorites serv...
1,TX00001368,T000265,C000004,196.33,GBP,2024-01-14 07:00:00,7,Sunday,MOBILE,2,OP58,DEV-E604,B6001,84120,0,5G,462cd323,REF51086615,208a1fbe-60ce-447a-b700-daab3da0ca42,91,5942,M00134,PLACE,regular general item,HOU,MOB,SN984297340,1.1.5,STORE003,HOU,30.04,15.020000,14.736105,4.60,25.44,20.84,unfoldiin gamber,1,unfoldiin gamber regular general item hou,5.284877,1.019265,0.153008,5,20,16,98.115942,0.693695,1,0.846988,1,1,1,30.04,place regular general item hou unfoldiin gamber,regular general item unfoldiin gamber,place regular general item unfoldiin gamber
2,TX00000422,T000083,C000171,172.16,USD,2024-08-09 13:00:00,13,Friday,CARD,3,OP23,DEV-C642,B5884,363382,0,WIFI,7fd24097,REF38935409,e1fbc784-1fcc-4816-93ab-e8244540b484,7,5651,M00045,LOCATION,regular basic basic basic regular,PHX,POS,SN605120084,3.9.49,OUTLET002,PHX,193.01,64.336667,23.284236,39.34,85.41,46.07,unit part stuff element part,0,unit part stuff element part regular basic bas...,5.154216,2.763100,1.121108,8,33,28,57.367544,0.327543,0,0.121108,0,1,3,193.01,location regular basic basic basic regular phx...,regular basic basic basic regular unit part st...,location regular basic basic basic regular uni...
3,TX00000413,T000078,C000450,465.80,GBP,2024-08-16 23:00:00,23,Friday,CARD,4,OP12,DEV-C374,B9213,479281,0,4G,3f2021da,REF45112407,73532bfb-91c0-4374-bd6f-e5321a16735a,81,5651,M00041,PLACE,regular product piece general,HOU,ONL,SN285112125,4.4.19,A3,HOU,145.51,36.377500,37.826970,2.65,81.99,79.34,unit part stuff meatowboudn unit part stuff uo...,0,unit part stuff meatowboudn unit part stuff uo...,6.145901,0.961682,0.312387,5,29,65,116.420895,1.253831,0,0.687611,0,2,2,145.51,place regular product piece general hou unit p...,regular product piece general unit part stuff ...,place regular product piece general unit part ...
4,TX00000451,T000086,C000486,425.81,EUR,2024-05-13 22:00:00,22,Monday,CARD,5,OP27,DEV-C722,B9005,376123,0,LAN,ceab216a,REF59097411,395f6a67-83dc-4675-ab2a-3d60ea698ca3,5,5651,M00046,SHOP,basic service common basic in basic,NYC,ONL,SN205183339,3.6.21,C1,NYC,214.67,42.934000,27.201423,10.89,70.27,59.38,element part ashfll serenidy vei part averwind...,0,element part ashfll serenidy vei part averwind...,6.056339,1.578373,0.504145,4,35,83,85.144971,0.636683,0,0.495854,0,2,2,214.67,shop basic service common basic in basic nyc e...,basic service common basic in basic element pa...,shop basic service common basic in basic eleme...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...

In [7]:
X_train_main = df[df.terminal_city.isin(['HOU', 'PHX', 'CHI'])]
X_test_main = df[~df.terminal_city.isin(['HOU', 'PHX', 'CHI'])]

y_train_main = X_train_main['true_mcc']
y_test_main = X_test_main['true_mcc']

### --------------------------------------- LogisticRegression + TF-IDF char serarated (0.5936 GroupCV)----------------------------------------

In [ ]:
numeric_features = [
    'items_total_price','items_mean_price','items_price_std','items_min_price',
    'items_max_price','items_price_range','items_vs_amount', 'amount_log',
    'terminal_name_len','terminal_desc_len','items_text_len','amount_per_item', 'items_price_skew', 
]


In [ ]:
numeric_features = [
    'items_total_price','items_mean_price','items_price_std','items_min_price',
    'items_max_price','items_price_range','items_vs_amount', 'amount_log',
    'terminal_name_len','terminal_desc_len','items_text_len','amount_per_item', 'items_price_skew', 
]

text_union = FeatureUnion([
    ('term_name', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,3), min_df=1, max_df=0.1)),
    ('term_desc', TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=1, max_df=0.1)),
    ('items', TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=1, max_df=0.1))
])


full_pipeline = FeatureUnion([
    ('text', Pipeline([
        ('selector', FunctionTransformer(lambda X: X['full_text'], validate=False)),
        ('tfidf', text_union)
    ])),
    ('numeric', Pipeline([
        ('selector', FunctionTransformer(lambda X: X[numeric_features], validate=False)),
        ('scaler', StandardScaler())
    ]))
])
model_lr_char = Pipeline([
    ('features', full_pipeline),
    ('clf', LogisticRegression(C=2.5, multi_class='multinomial', max_iter=2000, solver='saga'))
])

In [19]:
model_lr_char = Pipeline([
    ('features', full_pipeline),
    ('clf', LogisticRegression(C=2.5, multi_class='multinomial', max_iter=2000, solver='saga'))
])

In [ ]:
model_lr_char.fit(X_train_main, y_train_main)

NameError: name 'X_train_main' is not defined

In [611]:
preds_lr_char_main = model_lr_char.predict(X_test_main)
probs = model_lr_char.predict_proba(X_test_main).max(axis=1)


In [612]:
acc = accuracy_score(y_test_main, preds_lr_char_main)
print(f'Accuracy: {acc:.4f}')

Accuracy: 0.8877


In [ ]:
5470

In [9]:
X_random = df[numeric_features + ['full_text', 'terminal_name', 'terminal_description', 'items_text', 'char_text', 'word_text']]
y_random = df['true_mcc']

In [10]:
X1, X2, y1, y2 = train_test_split(X_random, y_random, test_size=0.2, random_state=42, stratify=y_random)

model_lr_char.fit(X1, y1)

NameError: name 'model_lr_char' is not defined

In [613]:
preds_lr_char_random = model_lr_char.predict(X2)
acc = accuracy_score(y2, preds_lr_char_random)
print(f'Accuracy: {acc:.4f}')

Accuracy: 0.8233


In [616]:
groups = df['terminal_name']
y_lr_cv = df['true_mcc']
X_lr_cv = df[numeric_features + ['full_text']]
gkf = GroupKFold(n_splits=4)
scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_lr_cv, y_lr_cv , groups=groups)):
    print(f"\nFold {fold+1}")
    X_train_txt = X_lr_cv.loc[train_idx]
    X_val_txt = X_lr_cv.loc[val_idx]
    y_train_fold = y_lr_cv[train_idx]
    y_val_fold   = y_lr_cv[val_idx]

#     text_union = FeatureUnion([
#     ('term_name', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,3), min_df=1, max_df=0.1)),
#     ('term_desc', TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=1, max_df=0.1)),
#     ('items', TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=1, max_df=0.1))
#     ])

#     full_pipeline = FeatureUnion([
#     ('text', Pipeline([
#         ('selector', FunctionTransformer(lambda X: X['full_text'], validate=False)),
#         ('tfidf', text_union)
#     ])),
#     ('numeric', Pipeline([
#         ('selector', FunctionTransformer(lambda X: X[numeric_features], validate=False)),
#         ('scaler', StandardScaler())
#     ]))
# ])

#     model_lr_char = Pipeline([
#     ('features', full_pipeline),
#     ('clf', LogisticRegression(C=2.5, multi_class='multinomial', max_iter=2000, solver='saga'))
# ])

    model_lr_char.fit(X_train_txt, y_train_fold)
    val_preds = model_lr_char.predict(X_val_txt)

    train_cities = set(groups.iloc[train_idx])
    val_cities   = set(groups.iloc[val_idx])


    acc_fold = accuracy_score(y_val_fold, val_preds)
    print(f"Fold {fold+1} ensemble accuracy: {acc_fold:.4f}")
    
    # print(f"Train cities ({len(train_cities)}): {sorted(train_cities)}")
    # print(f"Val cities   ({len(val_cities)}): {sorted(val_cities)}")
    scores.append(acc_fold)

print(f"\nGroupCV ensemble accuracy: {np.mean(scores):.4f}")


Fold 1


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 1 ensemble accuracy: 0.0635

Fold 2


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 2 ensemble accuracy: 0.5576

Fold 3


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 3 ensemble accuracy: 0.6633

Fold 4


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 4 ensemble accuracy: 0.6903

GroupCV ensemble accuracy: 0.4937


### ---------------------------------------------- LogisticRegression + TF-IDF word (0.49 GroupCV) ------------------------------------------

In [22]:
word_tfidf = TfidfVectorizer(analyzer="word", ngram_range=(1,2), max_features=5000, min_df=1, max_df=0.01)
char_tfidf = TfidfVectorizer(analyzer="char", ngram_range=(3,5), max_features=7000, min_df=1, max_df=0.01)
text_vectorizer = FeatureUnion([("word", word_tfidf), ("char", char_tfidf)])

word_pipeline = Pipeline([
    ('selector', FunctionTransformer(lambda X: X['word_text'], validate=False)),
    ('tfidf', text_vectorizer)
])

full_pipeline = FeatureUnion([
    ('word_text', word_pipeline),
    ('numeric', Pipeline([
        ('selector', FunctionTransformer(lambda X: X[numeric_features], validate=False)),
        ('scaler', StandardScaler())
    ]))
])

model_lr_word = Pipeline([
    ('features', full_pipeline),
    ('clf', LogisticRegression(
        C=15,
        multi_class='multinomial',
        solver='saga',
        max_iter=2000,
        n_jobs=-1
    ))
])


In [23]:
model_lr_word.fit(X_train_main, y_train_main)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,steps,"[('features', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformer_list,"[('word_text', ...), ('numeric', ...)]"
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,func,<function <la...t 0x15c668ea0>
,inverse_func,None


In [24]:
preds_lr_word_main = model_lr_word.predict(X_test_main)
probas = model_lr_word.predict_proba(X_test_main).max(axis=1)

accur = accuracy_score(y_test_main, preds_lr_word_main)
accur

0.34477379095163807

In [25]:
model_lr_word.fit(X1, y1)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,steps,"[('features', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformer_list,"[('word_text', ...), ('numeric', ...)]"
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,func,<function <la...t 0x15c668ea0>
,inverse_func,None


In [26]:
preds_lr_word_random = model_lr_word.predict(X2)
acc_lr_word_random = accuracy_score(y2, preds_lr_word_random)
acc_lr_word_random

0.8133333333333334

In [28]:
groups = df['terminal_city']
y_lr_cv_2 = df['true_mcc']
X_lr_cv_2 = df[numeric_features + ['word_text']]
gkf = GroupKFold(n_splits=5)
scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_lr_cv_2, y_lr_cv_2 , groups=groups)):
    print(f"\nFold {fold+1}")
    X_train_txt = X_lr_cv_2.loc[train_idx]
    X_val_txt = X_lr_cv_2.loc[val_idx]
    y_train_fold = y_lr_cv_2[train_idx]
    y_val_fold   = y_lr_cv_2[val_idx]

    # word_tfidf = TfidfVectorizer(analyzer="word", ngram_range=(1,2), max_features=5000, min_df=1, max_df=0.01)
    # char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), max_features=7000, min_df=1, max_df=0.01)
    # text_vectorizer = FeatureUnion([("word", word_tfidf), ("char", char_tfidf)])

    # word_pipeline = Pipeline([
    # ('selector', FunctionTransformer(lambda X: X['word_text'], validate=False)),
    # ('tfidf', text_vectorizer)
    # ])

    # full_pipeline = FeatureUnion([
    # ('word_text', word_pipeline),
    # ('numeric', Pipeline([
    #     ('selector', FunctionTransformer(lambda X: X[numeric_features], validate=False)),
    #     ('scaler', StandardScaler())
    # ]))
    # ])

    # model_lr_word = Pipeline([
    # ('features', full_pipeline),
    # ('clf', LogisticRegression(
    #     C=20.0,
    #     multi_class='multinomial',
    #     solver='saga',
    #     max_iter=2000,
    #     n_jobs=-1
    # ))
    # ])

    model_lr_word.fit(X_train_txt, y_train_fold)
    val_preds = model_lr_word.predict(X_val_txt)

    train_cities = set(groups.iloc[train_idx])
    val_cities   = set(groups.iloc[val_idx])


    acc_fold = accuracy_score(y_val_fold, val_preds)
    print(f"Fold {fold+1} ensemble accuracy: {acc_fold:.4f}")
    
    print(f"Train cities ({len(train_cities)}): {sorted(train_cities)}")
    print(f"Val cities   ({len(val_cities)}): {sorted(val_cities)}")
    scores.append(acc_fold)

print(f"\nGroupCV ensemble accuracy: {np.mean(scores):.4f}")


Fold 1


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 1 ensemble accuracy: 0.3975
Train cities (4): ['CHI', 'HOU', 'LA', 'PHX']
Val cities   (1): ['NYC']

Fold 2


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 2 ensemble accuracy: 0.4984
Train cities (4): ['CHI', 'HOU', 'NYC', 'PHX']
Val cities   (1): ['LA']

Fold 3


...

Fold 3 ensemble accuracy: 0.4643
Train cities (4): ['CHI', 'LA', 'NYC', 'PHX']
Val cities   (1): ['HOU']

Fold 4


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 4 ensemble accuracy: 0.4983
Train cities (4): ['CHI', 'HOU', 'LA', 'NYC']
Val cities   (1): ['PHX']

Fold 5


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 5 ensemble accuracy: 0.5476
Train cities (4): ['HOU', 'LA', 'NYC', 'PHX']
Val cities   (1): ['CHI']

GroupCV ensemble accuracy: 0.4812


In [723]:
from scipy.stats import pearsonr

err1 = (preds_lr_char_main != y_test_main).astype(int)
err2 = (preds_lr_word_main != y_test_main).astype(int)

print("Error correlation:", pearsonr(err1, err2)[0])

Error correlation: 0.5551038660908324


### ----------------------------------------------- LinearSVC + TF-IDF char ----------------------------------------------------------------

In [11]:
text_col = ['char_text']

numeric_features = [
    'items_total_price','items_mean_price','items_price_std',
    'items_min_price','items_max_price','items_price_range',
    'items_vs_amount','amount_log',
    'terminal_name_len','terminal_desc_len',
    'items_text_len','amount_per_item','items_price_skew'
]

X_svc = df[text_col + numeric_features]
y_svc = df['true_mcc']
groups = df['terminal_city']


In [ ]:
text_col = ['char_text']

numeric_features = [
    'items_total_price','items_mean_price','items_price_std',
    'items_min_price','items_max_price','items_price_range',
    'items_vs_amount','amount_log',
    'terminal_name_len','terminal_desc_len',
    'items_text_len','amount_per_item','items_price_skew'
]

X_svc = df[text_col + numeric_features]

text_transformer = Pipeline([
    ('select', FunctionTransformer(lambda x: x['char_text'], validate=False)),
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2,5),
        max_features=7000,
        min_df=1,
        max_df=0.1,
        sublinear_tf=True
    ))
])

features = ColumnTransformer(
    transformers=[
        ('char', text_transformer, text_col),
        ('num', StandardScaler(), numeric_features)
    ]
)

svc = LinearSVC(C=1.5, max_iter=4000)

calibrated_svc = CalibratedClassifierCV(
    estimator=svc,
    cv=3,
    method='sigmoid'
)

model_svc = Pipeline([
    ('features', features),
    ('clf', calibrated_svc)
])


In [12]:
text_transformer = Pipeline([
    ('select', FunctionTransformer(lambda x: x['char_text'], validate=False)),
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2,5),
        max_features=7000,
        min_df=1,
        max_df=0.1,
        sublinear_tf=True
    ))
])

features = ColumnTransformer(
    transformers=[
        ('char', text_transformer, text_col),
        ('num', StandardScaler(), numeric_features)
    ]
)

svc = LinearSVC(C=1.5, max_iter=4000)

calibrated_svc = CalibratedClassifierCV(
    estimator=svc,
    cv=3,
    method='sigmoid'
)

model_svc = Pipeline([
    ('features', features),
    ('clf', calibrated_svc)
])


In [13]:
x_svc_1, x_svc_2, y_svc_1, y_svc_2 = train_test_split(X_svc, y_svc, test_size=0.2, random_state=42, stratify=y_svc)
model_svc.fit(x_svc_1, y_svc_1)
preds_svc = model_svc.predict(x_svc_2)
acc_svc = accuracy_score(y_svc_2, preds_svc)
acc_svc

0.87

In [14]:
model_svc.fit(X_train_main, y_train_main)

,steps,"[('features', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('char', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:
preds_svc_main = model_svc.predict(X_test_main)
acc_svc_main = accuracy_score(y_test_main, preds_svc_main)
acc_svc_main

0.6146645865834633

In [16]:
gkf = GroupKFold(n_splits=4)
scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_svc, y_svc, groups)):
    print(f"\nFold {fold+1}")

    X_train = X_svc.iloc[train_idx]
    X_val   = X_svc.iloc[val_idx]
    y_train = y_svc.iloc[train_idx]
    y_val   = y_svc.iloc[val_idx]

    model_svc.fit(X_train, y_train)

    val_preds = model_svc.predict(X_val)
    acc = accuracy_score(y_val, val_preds)

    print(f"Fold {fold+1} accuracy: {acc:.4f}")
    scores.append(acc)

print(f"\nGroupCV accuracy: {np.mean(scores):.4f}")



Fold 1
Fold 1 accuracy: 0.6646

Fold 2
Fold 2 accuracy: 0.7085

Fold 3
Fold 3 accuracy: 0.6591

Fold 4
Fold 4 accuracy: 0.6443

GroupCV accuracy: 0.6691


что дальше?? 

- обучить цвс на нумерик 
- собрать линейный ансамбль 
- добавить в ансамбль катбуст 
- попробовать мета модель 

щас пока только на 3 модельках потом мб 4 добавим 

- сделать по софт вотинг 
- сделать мета модель 
- акураси на груп цв 


### ----------------------------------------------------Ensemble(lr+lr+svc)---------------------------------------------------------------------

In [29]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score
import numpy as np

groups = df['terminal_city']
y = df['true_mcc']
X = df[numeric_features + ['char_text', 'word_text', 'terminal_name', 'terminal_description', 'items_text', 'full_text']]

le = LabelEncoder()
y_encoded = le.fit_transform(y)


gkf = GroupKFold(n_splits=5)

scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_encoded, groups)):
    print(f"\nFold {fold+1}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold = y_encoded[train_idx]
    y_val_fold   = y_encoded[val_idx]

    X_train_svc = X_train[['char_text'] + numeric_features]
    X_test_svc = X_val[['char_text'] + numeric_features]
    X_train_lr_char = X_train[numeric_features + ['terminal_name', 'terminal_description', 'items_text', 'full_text']]
    X_test_lr_char = X_val[numeric_features + ['terminal_name', 'terminal_description', 'items_text', 'full_text']]
    X_train_lr_word = X_train[numeric_features + ['word_text']]
    X_test_lr_word = X_val[numeric_features + ['word_text']]

    # ========= 1. fit models =========
    model_svc.fit(X_train_svc, y_train_fold)
    model_lr_char.fit(X_train_lr_char, y_train_fold)
    model_lr_word.fit(X_train_lr_word, y_train_fold)

    # ========= 2. predict_proba =========
    p_svc     = model_svc.predict_proba(X_test_svc)
    p_lr_char = model_lr_char.predict_proba(X_test_lr_char)
    p_lr_word = model_lr_word.predict_proba(X_test_lr_word)

    # ========= 3. ensemble =========
    p_ensemble = (
        0.6 * p_svc +
        0.25 * p_lr_char +
        0.15 * p_lr_word
    )

    y_pred = p_ensemble.argmax(axis=1)
    y_pred_mcc = le.inverse_transform(y_pred) 

    acc = accuracy_score(y_val_fold, y_pred)
    scores.append(acc)

    print(f"Fold {fold+1} accuracy: {acc:.4f}")
    print("Val cities:", sorted(set(groups.iloc[val_idx])))

print(f"\nGroupCV ensemble accuracy: {np.mean(scores):.4f}")



Fold 1


NameError: name 'model_lr_char' is not defined

попробовать на рандом сплит 

In [754]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [755]:
X_ens_train, X_ens_test, y_ens_train, y_ens_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
model_svc.fit(X_ens_train, y_ens_train)
model_lr_char.fit(X_ens_train, y_ens_train)
model_lr_word.fit(X_ens_train, y_ens_train) 
p_svc     = model_svc.predict_proba(X_ens_test)
p_lr_char = model_lr_char.predict_proba(X_ens_test)
p_lr_word = model_lr_word.predict_proba(X_ens_test)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [757]:
p_ensemble = (
        0.65 * p_svc +
        0.25 * p_lr_char +
        0.10 * p_lr_word
    )

y_pred = p_ensemble.argmax(axis=1)
y_pred_mcc = le.inverse_transform(y_pred)

acc = accuracy_score(y_ens_test, y_pred)
acc

0.86

In [759]:
groups = df['terminal_city']

classes = np.unique(y)
n_classes = len(classes)

In [761]:
gkf = GroupKFold(n_splits=5)

oof_svc      = np.zeros((len(X), n_classes))
oof_lr_char  = np.zeros((len(X), n_classes))
oof_lr_word  = np.zeros((len(X), n_classes))

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y_encoded, groups)):
    print(f"\nFold {fold+1}")

    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y_encoded[tr_idx], y_encoded[val_idx]

    # ===== 1. LinearSVC (char + numeric)
    model_svc.fit(X_tr, y_tr)
    oof_svc[val_idx] = model_svc.predict_proba(X_val)

    # ===== 2. LR char
    model_lr_char.fit(X_tr, y_tr)
    oof_lr_char[val_idx] = model_lr_char.predict_proba(X_val)

    # ===== 3. LR word
    model_lr_word.fit(X_tr, y_tr)
    oof_lr_word[val_idx] = model_lr_word.predict_proba(X_val)

    # sanity check
    print(
        "SVC:", accuracy_score(y_val, oof_svc[val_idx].argmax(1)),
        "LR_char:", accuracy_score(y_val, oof_lr_char[val_idx].argmax(1)),
        "LR_word:", accuracy_score(y_val, oof_lr_word[val_idx].argmax(1))
    )



Fold 1


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


SVC: 0.6645962732919255 LR_char: 0.5434782608695652 LR_word: 0.40062111801242234

Fold 2


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


SVC: 0.7084639498432602 LR_char: 0.6363636363636364 LR_word: 0.49843260188087773

Fold 3


...

SVC: 0.6590909090909091 LR_char: 0.6103896103896104 LR_word: 0.4642857142857143

Fold 4


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


SVC: 0.6421404682274248 LR_char: 0.6120401337792643 LR_word: 0.4983277591973244

Fold 5


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


SVC: 0.6904761904761905 LR_char: 0.6587301587301587 LR_word: 0.5476190476190477


catboost

In [17]:
text_features = ['full_text'] 
num_features = numeric_features
X_cat = df[num_features + text_features]


def build_cat_model():
    return CatBoostClassifier(
        iterations=400,
        depth=6,
        learning_rate=0.1,
        loss_function='MultiClass',
        eval_metric='Accuracy',
        random_seed=42,
        verbose=False
    )


In [ ]:
oof_cat = np.zeros((len(df), n_classes))

In [38]:
gkf = GroupKFold(n_splits=5)
scores = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_cat, y, groups)):
    print(f"CatBoost fold {fold+1}")

    X_tr, X_val = X_cat.iloc[tr_idx], X_cat.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model_cat = build_cat_model()

    model_cat.fit(
        X_tr,
        y_tr,
        text_features=[X_cat.columns.get_loc('full_text')]
    )

    y_pred = model_cat.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    print(f"Fold {fold+1} accuracy: {acc:.4f}")




CatBoost fold 1
Fold 1 accuracy: 0.1522
CatBoost fold 2
Fold 2 accuracy: 0.2163
CatBoost fold 3
Fold 3 accuracy: 0.1234
CatBoost fold 4
Fold 4 accuracy: 0.2040
CatBoost fold 5
Fold 5 accuracy: 0.1825


In [18]:
model_cat = CatBoostClassifier(
        iterations=400,
        depth=6,
        learning_rate=0.1,
        loss_function='MultiClass',
        eval_metric='Accuracy',
        random_seed=42,
        verbose=False
    )

## !!!!!!!!!!

In [39]:
groups = df['terminal_city']
y = df['true_mcc']
X = df[numeric_features + ['char_text', 'word_text', 'terminal_name', 'terminal_description', 'items_text', 'full_text']]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

gkf = GroupKFold(n_splits=5)

scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_encoded, groups)):
    print(f"\nFold {fold+1}")

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_encoded[train_idx], y_encoded[val_idx]

    X_tr_cat = X_tr[numeric_features + ['full_text']]
    X_val_cat = X_val[numeric_features + ['full_text']]


    # ========= 1. fit models =========
    model_svc.fit(X_tr, y_tr)
    model_cat.fit(
        X_tr_cat,
        le.inverse_transform(y_tr),
        text_features=[X_tr_cat.columns.get_loc('full_text')]
    )
    model_lr_word.fit(X_tr, y_tr)

    # ========= 2. predict_proba =========
    p_svc     = model_svc.predict_proba(X_val)
    p_cat     = model_cat.predict_proba(X_val_cat)
    p_lr_char = model_lr_word.predict_proba(X_val)

    # ========= 3. ensemble =========
    p_ensemble = (
        0.6 * p_svc +
        0.15 * p_cat +
        0.25 * p_lr_char
    )

    y_pred = p_ensemble.argmax(axis=1)
    y_pred_mcc = le.inverse_transform(y_pred) 

    acc = accuracy_score(y_val, y_pred)
    scores.append(acc)

    print(f"Fold {fold+1} accuracy: {acc:.4f}")
    print("Val cities:", sorted(set(groups.iloc[val_idx]))) 
print(f"\nGroupCV ensemble accuracy: {np.mean(scores):.4f}")



Fold 1


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 1 accuracy: 0.6398
Val cities: ['NYC']

Fold 2


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 2 accuracy: 0.7022
Val cities: ['LA']

Fold 3


...

Fold 3 accuracy: 0.6623
Val cities: ['HOU']

Fold 4


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 4 accuracy: 0.6589
Val cities: ['PHX']

Fold 5


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 5 accuracy: 0.6944
Val cities: ['CHI']

GroupCV ensemble accuracy: 0.6715


In [ ]:
X_tr_random, X_val_random, y_tr_rand, y_val_rand = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

X_tr_cat = X_tr_random[numeric_features + ['full_text']]
X_val_cat = X_val_random[numeric_features + ['full_text']]

model_svc.fit(X_tr_random, y_tr_rand)
model_cat.fit(
        X_tr_cat,
        le.inverse_transform(y_tr_rand),
        text_features=[X_tr_cat.columns.get_loc('full_text')]
    )
model_lr_word.fit(X_tr_random, y_tr_rand)

    # ========= 2. predict_proba =========
p_svc     = model_svc.predict_proba(X_val_random)
p_cat     = model_cat.predict_proba(X_val_cat)
p_lr_char = model_lr_word.predict_proba(X_val_random)

    # ========= 3. ensemble =========
p_ensemble = (
        0.6 * p_svc +
        0.25 * p_cat +
        0.15 * p_lr_char
    )

y_pred = p_ensemble.argmax(axis=1)
y_pred_mcc = le.inverse_transform(y_pred) 

acc = accuracy_score(y_val_rand, y_pred)
scores.append(acc)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [35]:
acc

0.9366666666666666

In [ ]:
# итого можем получить не более 67 я хз что еще можно сделать то есть по сути это у тебя щас наверно уже итговая модель если не улучшим 
# завтра сохранить все веса и писать апи и докер 
# НЕ ЗАБЫТЬ ПРОВЕРКИ НА НЕОРДИНАРНЫЕ СЛУЧАИ 


SyntaxError: invalid syntax (4232510701.py, line 1)

In [ ]:
0.6751

In [813]:
X_meta = np.hstack([
    oof_svc,
    oof_cat
])
X_meta.shape

(1500, 30)

In [814]:

meta_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.05
))
])


In [815]:
meta_scores = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_meta, y, groups)):
    X_tr, X_val = X_meta[tr_idx], X_meta[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    meta_model.fit(X_tr, y_tr)
    preds = meta_model.predict(X_val)

    acc = accuracy_score(y_val, preds)
    meta_scores.append(acc)

    print(f"Fold {fold+1} stacking acc: {acc:.4f}")

print(f"\nOOF stacking GroupCV accuracy: {np.mean(meta_scores):.4f}")


Fold 1 stacking acc: 0.6211
Fold 2 stacking acc: 0.6865
Fold 3 stacking acc: 0.6558
Fold 4 stacking acc: 0.6321
Fold 5 stacking acc: 0.6944

OOF stacking GroupCV accuracy: 0.6580


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Pytho

In [781]:
coef = meta_model.coef_      
n_classes = coef.shape[0]

model_names = [
    "svc_char",
    "lr_char",
    "lr_word",
    "catboost"
]

rows = []

for i, name in enumerate(model_names):
    start = i * n_classes
    end   = (i + 1) * n_classes

    block = coef[:, start:end]

    rows.append({
        "model": name,
        "mean_abs_coef": np.mean(np.abs(block)),
        "max_abs_coef":  np.max(np.abs(block)),
        "std_coef":      np.std(block)
    })

coef_df = pd.DataFrame(rows).sort_values("mean_abs_coef", ascending=False)
coef_df


,model,mean_abs_coef,max_abs_coef,std_coef
0,svc_char,0.641889,4.939742,1.133295
3,catboost,0.538799,3.132010,0.723323
2,lr_word,0.454586,2.497928,0.605940
1,lr_char,0.392373,2.299030,0.567855


обучение для продакшена 

In [20]:
X_cat_final = df[numeric_features + ['full_text']]

In [22]:
model_cat = CatBoostClassifier(
        iterations=400,
        depth=6,
        learning_rate=0.1,
        loss_function='MultiClass',
        eval_metric='Accuracy',
        random_seed=42,
        verbose=False
    )

In [23]:
numeric_features = [
    'items_total_price','items_mean_price','items_price_std','items_min_price',
    'items_max_price','items_price_range','items_vs_amount', 'amount_log',
    'terminal_name_len','terminal_desc_len','items_text_len','amount_per_item', 'items_price_skew', 
]

In [48]:
def select_full_text(X):
    return X['full_text']

def select_numeric(X):
    return X[numeric_features]

In [49]:
text_union = FeatureUnion([
    ('term_name', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,3), min_df=1, max_df=0.1)),
    ('term_desc', TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=1, max_df=0.1)),
    ('items', TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=1, max_df=0.1))
])

full_pipeline = FeatureUnion([
    ('text', Pipeline([
        ('selector', FunctionTransformer(select_full_text, validate=False)),
        ('tfidf', text_union)
    ])),
    ('numeric', Pipeline([
        ('selector', FunctionTransformer(select_numeric, validate=False)),
        ('scaler', StandardScaler())
    ]))
])

model_lr_char = Pipeline([
    ('features', full_pipeline),
    ('clf', LogisticRegression(
        C=2.5,
        multi_class='multinomial',
        max_iter=2000,
        solver='saga'
    ))
])


In [28]:
text_col = ['char_text']
X_svc_final = df[text_col + numeric_features]

In [41]:
def select_char_text(X):
    return X['char_text']


In [42]:
text_transformer = Pipeline([
    ('select', FunctionTransformer(select_char_text, validate=False)),
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2,5),
        max_features=7000,
        min_df=1,
        max_df=0.1,
        sublinear_tf=True
    ))
])


features = ColumnTransformer(
    transformers=[
        ('char', text_transformer, text_col),
        ('num', StandardScaler(), numeric_features)
    ]
)

svc = LinearSVC(C=1.5, max_iter=4000)

calibrated_svc = CalibratedClassifierCV(
    estimator=svc,
    cv=3,
    method='sigmoid'
)

model_svc = Pipeline([
    ('features', features),
    ('clf', calibrated_svc)
])


In [27]:
le = LabelEncoder()
y_raw_mcc = df['true_mcc']
y_encoded = le.fit_transform(y_raw_mcc)

In [50]:
model_lr_char.fit(df, y_encoded)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,steps,"[('features', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformer_list,"[('text', ...), ('numeric', ...)]"
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,func,<function sel...t 0x1782e9940>
,inverse_func,None


In [29]:
model_svc.fit(X_svc_final, y_encoded)
model_lr_char.fit(df, y_encoded)
model_cat.fit(X_cat_final, y_encoded, text_features=[X_cat_final.columns.get_loc('full_text')])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [ ]:
/Users/mila/emmi-ffjrr_gmail-com/solution/model

In [35]:
import joblib

In [33]:
model_cat.save_model("/Users/mila/emmi-ffjrr_gmail-com/solution/model/model_cat.cbm")


In [44]:
joblib.dump(model_lr_char, "/Users/mila/emmi-ffjrr_gmail-com/solution/model/model_lr_char.pkl")


['/Users/mila/emmi-ffjrr_gmail-com/solution/model/model_lr_char.pkl']

In [45]:
joblib.dump(model_svc, "/Users/mila/emmi-ffjrr_gmail-com/solution/model/model_svc.pkl")

['/Users/mila/emmi-ffjrr_gmail-com/solution/model/model_svc.pkl']

In [46]:
joblib.dump(le, "/Users/mila/emmi-ffjrr_gmail-com/solution/model/le_mcc.pkl")

['/Users/mila/emmi-ffjrr_gmail-com/solution/model/le_mcc.pkl']

In [47]:
df["full_text_len"] = df["full_text"].str.len()

df["full_text_len"].describe(percentiles=[0.95, 0.99])


count    1500.000000
mean       69.602667
std        18.977976
min        28.000000
50%        68.000000
95%       104.050000
99%       117.000000
max       135.000000
Name: full_text_len, dtype: float64

In [52]:
req = {
    "transactions": [
        {
            "transaction_id": "TX00001",
            "terminal_name": "STORE001",
            "terminal_description": "common common",
            "city": "NYC",
            "amount": 123.45,
            "items": [{"name": "item1", "price": 50}, {"name": "item2", "price": 73.45}]
        }
    ]
}

In [53]:
import pandas as pd
import re
from typing import List, Dict, Any
import numpy as np

numeric_features = [
    'items_total_price','items_mean_price','items_price_std','items_min_price',
    'items_max_price','items_price_range','items_vs_amount', 'amount_log',
    'terminal_name_len','terminal_desc_len','items_text_len','amount_per_item', 'items_price_skew', 
]

max_text_len = 150

class BadRequestError(Exception):
    pass

def select_char_text(X):
    return X['char_text']

def select_full_text(X):
    return X['full_text']

def select_numeric(X):
    return X[numeric_features]

def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_text_len]

def safe_text(value: Any, max_len: int = 10_000) -> str:

    if value is None:
        return ""
    
    text = str(value)
    return text[:max_len]

def build_chars_text(tx: Dict[str, Any]) -> str:
    parts: List[str] = []

    if tx.get("terminal_name"):
        parts.append(tx["terminal_name"])

    if tx.get("terminal_description"):
        parts.append(tx["terminal_description"])

    items = tx.get("items", [])
    if isinstance(items, list):
        for item in items:
            name = item.get("name")
            if isinstance(name, str):
                parts.append(name)

    raw_text = " ".join(parts)
    return normalize_text(raw_text)

def build_full_text(tx: Dict[str, Any]) -> str:
    parts: List[str] = []

    if tx.get("terminal_name"):
        parts.append(tx["terminal_name"])

    if tx.get("terminal_description"):
        parts.append(tx["terminal_description"])

    if tx.get("city"):
        parts.append(tx["city"])

    items = tx.get("items", [])
    if isinstance(items, list):
        for item in items:
            name = item.get("name")
            if isinstance(name, str):
                parts.append(name)

    raw_text = " ".join(parts)
    return normalize_text(raw_text)


def build_numeric_features(transactions: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []

    for tx in transactions:
        items = tx.get("items")                 

        if not isinstance(items, list) or len(items) == 0:
            raise BadRequestError("items must be a non-empty list")

        prices = []
        item_texts = []

        for item in items:
            if "price" not in item or "name" not in item:
                raise BadRequestError("each item must contain name and price")

            price = item["price"]

            if not isinstance(price, (int, float)) or price <= 0:
                raise BadRequestError("item price must be positive number")

            prices.append(price)
            item_texts.append(str(item["name"]))

        prices = np.array(prices, dtype=float)

        amount = tx.get("amount")
        if not isinstance(amount, (int, float)) or amount <= 0:
            raise BadRequestError("amount must be positive number")

        item_count = len(prices)

        items_total_price = prices.sum()
        items_mean_price = prices.mean()
        items_min_price = prices.min()
        items_max_price = prices.max()
        items_price_std = prices.std(ddof=0)
        items_price_range = items_max_price - items_min_price

        amount_per_item = amount / (item_count + 1e-3)
        items_vs_amount = items_total_price / amount
        amount_log = np.log1p(amount)
        items_price_skew = (
            (items_max_price - items_mean_price)
            / (items_mean_price + 1e-3)
        )

        terminal_name = tx.get("terminal_name") or ""
        terminal_desc = tx.get("terminal_description") or ""

        rows.append({
            "items_total_price": items_total_price,
            "items_mean_price": items_mean_price,
            "items_price_std": items_price_std,
            "items_min_price": items_min_price,
            "items_max_price": items_max_price,
            "items_price_range": items_price_range,
            "items_vs_amount": items_vs_amount,
            "amount_log": amount_log,
            "terminal_name_len": len(str(terminal_name)),
            "terminal_desc_len": len(str(terminal_desc)),
            "items_text_len": len(" ".join(item_texts)),
            "amount_per_item": amount_per_item,
            "items_price_skew": items_price_skew,
        })

    return pd.DataFrame(rows)

In [54]:
def prepare_data(payload: Dict[str, Any]) -> pd.DataFrame:
    if not isinstance(payload, dict):
        raise BadRequestError("payload must be a JSON object")

    transactions = payload.get("transactions")

    if not isinstance(transactions, list) or len(transactions) == 0:
        raise BadRequestError("transactions must be a non-empty list")

    char_texts = []
    full_texts = []

    for tx in transactions:
        if not isinstance(tx, dict):
            raise BadRequestError("each transaction must be an object")

        char_texts.append(build_chars_text(tx))
        full_texts.append(build_full_text(tx))

    numeric_df = build_numeric_features(transactions)

    df = numeric_df.copy()
    df["char_text"] = char_texts
    df["full_text"] = full_texts

    return df

In [56]:
dff = prepare_data(req)
dff

,items_total_price,items_mean_price,items_price_std,items_min_price,items_max_price,items_price_range,items_vs_amount,amount_log,terminal_name_len,terminal_desc_len,items_text_len,amount_per_item,items_price_skew,char_text,full_text
0,123.45,61.725,11.725,50.0,73.45,23.45,1.0,4.823904,8,13,11,61.694153,0.189952,store001 common common item1 item2,store001 common common nyc item1 item2
